# 📈 Sales Forecasting: Store-Item Demand (LightGBM vs Baselines vs ETS)

**Problem:** 10 stores x 50 items ka 5 saal (2013-2017) ka daily sales data hai. Hume **agle 90 din** ka sales forecast karna hai, har store-item ke liye (500 time series).

**Approach:**
1. Time-based split (**last 90 din = test**, usse pehle 90 din = validation). Random split nahi, kyunki time series me future ka data leak ho jata hai.
2. **Baselines:** Seasonal Naive, Smoothed Seasonal Naive, Smoothed Seasonal Naive + Growth
3. **Classical model:** Holt-Winters (ETS) total sales pe
4. **ML model:** ek **global LightGBM** (saari 500 series ek model me) jisme lag features sirf **>= 91 din** purane hain, isliye 90-din ka forecast **leak-free** hota hai (automatic check bhi hai)
5. Evaluation: **SMAPE** (competition metric), WAPE, MAE, RMSE + diagnostics (horizon, weekday, store, bias)
6. Final model se **Jan-Mar 2018** ka forecast

**Dataset:** https://www.kaggle.com/competitions/demand-forecasting-kernels-only (file: `train.csv`)

**Kaggle setup:**
- Competition page pe **Join Competition** (rules accept) karo
- Notebook me Add Input, Competition Data, `demand-forecasting-kernels-only`
- GPU ki zaroorat nahi, Internet bhi off chalega. Total run time ~15 min.

## 1. Setup & Config

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import lightgbm as lgb
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

# ------------------------------ CONFIG ------------------------------
SEED = 42
HORIZON = 90                      # kitne din aage forecast karna hai
MIN_LAG = HORIZON + 1             # features sirf itne purane (ya zyada) din se: leak-free
LAGS = [91, 98, 105, 112, 119, 126, 133, 140, 182, 350, 357, 364, 371, 378, 728]   # sab 7 ke multiple (same weekday)
ROLL_WINDOWS = [7, 14, 28, 56, 91, 182, 364]
LGB_PARAMS = dict(
    objective="regression", metric="l2", learning_rate=0.08, num_leaves=63,
    min_data_in_leaf=100, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=1.0, verbose=-1, seed=SEED,
)
MAX_ROUNDS = 2000
EARLY_STOP = 50
# --------------------------------------------------------------------

np.random.seed(SEED)
OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SEARCH_ROOTS = [r for r in ["/kaggle/input", "."] if os.path.exists(r)]
print("LightGBM:", lgb.__version__, "| Output dir:", OUT_DIR)

## 2. Load Data (path auto-detect)

In [ ]:
def find_train_csv():
    for root in SEARCH_ROOTS:
        for dirpath, _, files in os.walk(root):
            if "train.csv" in files:
                p = os.path.join(dirpath, "train.csv")
                cols = [c.lower() for c in pd.read_csv(p, nrows=0).columns]
                if {"date", "store", "item", "sales"} <= set(cols):
                    return p
    raise FileNotFoundError(
        "train.csv (date, store, item, sales) nahi mila. Competition join karke "
        "'demand-forecasting-kernels-only' ko Add Input karo."
    )

train_path = find_train_csv()
print("train.csv:", train_path)

df = pd.read_csv(train_path)
df.columns = [c.lower() for c in df.columns]
df = df[["date", "store", "item", "sales"]]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)

print("Shape:", df.shape)
df.head()

## 3. Data Quality Checks & Panel Build

In [ ]:
print("Missing values:\n", df.isna().sum().to_string())
print("\nDuplicate (date, store, item) rows:", int(df.duplicated(["date", "store", "item"]).sum()))
print("Date range:", df["date"].min().date(), "->", df["date"].max().date())
print("Stores:", df["store"].nunique(), "| Items:", df["item"].nunique())
print("Sales: min", df["sales"].min(), "| mean", round(df["sales"].mean(), 2),
      "| max", df["sales"].max(), "| zero-sales rows:", int((df["sales"] == 0).sum()))

hist_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
wide = df.set_index(["store", "item", "date"])["sales"].unstack("date").reindex(columns=hist_dates)
n_missing = int(wide.isna().sum().sum())
print(f"\nSeries: {wide.shape[0]} | Days: {wide.shape[1]} | missing store-item-days: {n_missing}")
if n_missing:
    print("Missing days ko 0 se fill kiya (sales nahi hui maan ke).")
    wide = wide.fillna(0)

SID = wide.index.to_frame(index=False)            # store, item (series order)
N_SERIES = len(SID)
S_hist = wide.values.astype("float64")             # (series, days)

# Future ke 90 din (NaN) jodte hain: baad me final forecast ke liye
future_dates = pd.date_range(hist_dates[-1] + pd.Timedelta(days=1), periods=HORIZON, freq="D")
all_dates = hist_dates.append(future_dates)
S_all = np.concatenate([S_hist, np.full((N_SERIES, HORIZON), np.nan)], axis=1)
N_HIST = len(hist_dates)
print("S_all shape (series, days incl. future):", S_all.shape)

## 4. Exploratory Data Analysis

In [ ]:
total = df.groupby("date")["sales"].sum()

fig, ax = plt.subplots(2, 1, figsize=(14, 8))
ax[0].plot(total.index, total.values, lw=0.6, color="#4C72B0", label="daily total")
ax[0].plot(total.index, total.rolling(30, center=True).mean(), color="crimson", lw=2, label="30-day mean")
ax[0].set_title("Total daily sales (all stores and items)"); ax[0].legend()

yearly = total.groupby(total.index.year).sum()
ax[1].bar(yearly.index.astype(str), yearly.values, color="#55A868")
for i, (y, v) in enumerate(yearly.items()):
    growth = "" if i == 0 else f"  ({100 * (v / yearly.iloc[i - 1] - 1):+.1f}%)"
    ax[1].text(i, v, f"{v / 1e6:.2f}M{growth}", ha="center", va="bottom", fontsize=9)
ax[1].set_title("Total sales per year (YoY growth)")
ax[1].set_xlabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eda_trend.png"), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

# Monthly seasonality per year
m = df.assign(year=df["date"].dt.year, month=df["date"].dt.month).groupby(["year", "month"])["sales"].sum().unstack(0)
m.plot(ax=ax[0], marker="o", lw=1.5)
ax[0].set_title("Monthly sales by year (yearly seasonality)")
ax[0].set_xlabel("Month"); ax[0].legend(title="Year", fontsize=8)

# Weekday pattern
dow = total.groupby(total.index.dayofweek).mean()
ax[1].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], dow.values, color="#C44E52")
ax[1].set_title("Average daily total by weekday (weekly seasonality)")

# Store ranking
st = df.groupby("store")["sales"].sum().sort_values()
ax[2].barh(st.index.astype(str), st.values, color="#8172B2")
ax[2].set_title("Total sales by store"); ax[2].set_ylabel("store")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eda_seasonality.png"), dpi=150)
plt.show()

print("Weekend vs weekday uplift: Sun/Mon ratio = %.2f" % (dow.iloc[6] / dow.iloc[0]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
it = df.groupby("item")["sales"].sum().sort_values()
pd.concat([it.head(5), it.tail(5)]).plot(kind="barh", ax=ax[0], color="#4C72B0")
ax[0].set_title("5 lowest and 5 highest selling items"); ax[0].set_ylabel("item")

heat = df.assign(month=df["date"].dt.month, year=df["date"].dt.year).groupby(["year", "month"])["sales"].sum().unstack()
sns.heatmap(heat / 1e3, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax[1], cbar_kws={"label": "sales (000s)"})
ax[1].set_title("Sales heatmap: year x month")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eda_items_heatmap.png"), dpi=150)
plt.show()

## 5. Validation Design (time-based)

```
|---------------- train ----------------|---- validation (90d) ----|---- test (90d) ----|
```

- **Test** = aakhri 90 din (final honest score, kabhi tuning me use nahi hota)
- **Validation** = test se pehle ke 90 din (LightGBM ke rounds choose karne ke liye)
- Final test forecast ke liye model **train + validation** pe dobara fit hota hai

In [ ]:
T_TEST = N_HIST - HORIZON            # test ka pehla column index
T_VAL = T_TEST - HORIZON             # validation ka pehla column
T_TRAIN0 = 364                       # pehla din jiska lag_364 available hai

train_idx = np.arange(T_TRAIN0, T_VAL)
val_idx = np.arange(T_VAL, T_TEST)
test_idx = np.arange(T_TEST, N_HIST)
future_idx = np.arange(N_HIST, N_HIST + HORIZON)

def span(idx):
    return f"{all_dates[idx[0]].date()} -> {all_dates[idx[-1]].date()} ({len(idx)} days)"

print("Train      :", span(train_idx))
print("Validation :", span(val_idx))
print("Test       :", span(test_idx))
print("Future     :", span(future_idx), "(labels nahi hain)")

fig, ax = plt.subplots(figsize=(14, 2.2))
ax.plot(total.index, total.values, color="lightgray", lw=0.6)
for idx, c, lab in [(train_idx, "#4C72B0", "train"), (val_idx, "#DD8452", "validation"), (test_idx, "#C44E52", "test")]:
    ax.axvspan(all_dates[idx[0]], all_dates[idx[-1]], color=c, alpha=0.35, label=lab)
ax.axvspan(all_dates[future_idx[0]], all_dates[future_idx[-1]], color="green", alpha=0.25, label="future forecast")
ax.legend(ncol=4, loc="upper left"); ax.set_yticks([]); ax.set_title("Time-based split")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "split.png"), dpi=150); plt.show()

## 6. Metrics

In [ ]:
def smape(a, f):
    # competition metric: 200 * mean(|f-a| / (|a|+|f|)), dono 0 ho toh error 0
    a, f = np.asarray(a, float).ravel(), np.asarray(f, float).ravel()
    denom = np.abs(a) + np.abs(f)
    out = np.where(denom == 0, 0.0, np.abs(f - a) / np.where(denom == 0, 1, denom))
    return 200 * out.mean()

def wape(a, f):
    a, f = np.asarray(a, float).ravel(), np.asarray(f, float).ravel()
    return 100 * np.abs(f - a).sum() / np.abs(a).sum()

def mae(a, f):
    return float(np.abs(np.asarray(a, float) - np.asarray(f, float)).mean())

def rmse(a, f):
    return float(np.sqrt(((np.asarray(a, float) - np.asarray(f, float)) ** 2).mean()))

A_test = S_all[:, test_idx]                     # actual test matrix (series x 90)
results = []

def record(name, F):
    # F: (series x 90) forecast matrix
    tot_a, tot_f = A_test.sum(axis=0), F.sum(axis=0)
    row = {"Model": name, "SMAPE": smape(A_test, F), "WAPE %": wape(A_test, F),
           "MAE": mae(A_test, F), "RMSE": rmse(A_test, F),
           "Total-level WAPE %": wape(tot_a, tot_f)}
    results.append(row)
    print(f"{name:34s} SMAPE {row['SMAPE']:6.2f} | WAPE {row['WAPE %']:6.2f}% | MAE {row['MAE']:6.2f} | "
          f"RMSE {row['RMSE']:6.2f} | total-level WAPE {row['Total-level WAPE %']:5.2f}%")
    return row

## 7. Feature Engineering (leak-free)

90 din aage forecast karna hai, isliye **har feature sirf `>= 91` din purane data se** banta hai
(lags, rolling means, year-ago neighbourhood, growth). Isse recursive forecasting ki zaroorat nahi padti.

| Group | Features |
|---|---|
| Lags | same-weekday lags: 91, 98 ... 140, 182, 350 ... 378, 728 |
| Rolling | mean (7..364 din) aur std(28), sab 91 din shifted |
| Same-weekday means | pichle 4 aur 8 haftey, year-ago +/- 2 haftey |
| Growth | recent 91-din level / year-ago level |
| Calendar | weekday, month, day, day-of-year, week, quarter, month start/end, year |
| IDs | store, item (categorical) |

In [ ]:
def build_features(S, dates):
    # S: (series, days) sales matrix (NaN = unknown). Return: dict name -> (series, days) float32 matrix.
    n, T = S.shape
    D = pd.DataFrame(S.T, index=dates)                  # days x series
    F = {}

    for L in LAGS:
        F[f"lag_{L}"] = D.shift(L).values.T

    for w in ROLL_WINDOWS:
        F[f"rm_{w}"] = D.rolling(w, min_periods=w).mean().shift(MIN_LAG).values.T
    F["rstd_28"] = D.rolling(28, min_periods=28).std().shift(MIN_LAG).values.T

    lag = lambda L: F[f"lag_{L}"]
    F["sdow_4"] = np.mean([lag(L) for L in (91, 98, 105, 112)], axis=0)
    F["sdow_8"] = np.mean([lag(L) for L in (91, 98, 105, 112, 119, 126, 133, 140)], axis=0)
    F["ya_5"] = np.mean([lag(L) for L in (350, 357, 364, 371, 378)], axis=0)

    rm91_prev = D.rolling(91, min_periods=91).mean().shift(MIN_LAG + 364).values.T
    with np.errstate(divide="ignore", invalid="ignore"):
        F["growth"] = np.where(rm91_prev > 0, F["rm_91"] / rm91_prev, np.nan)

    di = pd.DatetimeIndex(dates)
    cal = {
        "dow": di.dayofweek, "month": di.month, "day": di.day, "doy": di.dayofyear,
        "week": di.isocalendar().week.values.astype(int), "quarter": di.quarter,
        "is_month_start": di.is_month_start.astype(int), "is_month_end": di.is_month_end.astype(int),
        "year": di.year,
    }
    for k, v in cal.items():
        F[k] = np.broadcast_to(np.asarray(v)[None, :], (n, T))

    return {k: np.asarray(v, dtype="float32") for k, v in F.items()}

t0 = time.time()
FEATS = build_features(S_all, all_dates)
FEATURE_NAMES = list(FEATS.keys()) + ["store", "item"]
print(f"{len(FEATURE_NAMES)} features built in {time.time() - t0:.1f}s")

### Leakage check

Test (aur validation) period ke actual sales ko **NaN** karke features dobara banate hain.
Agar koi feature us period ke actual sales se bana hota, toh uski values badal jaati. Assert ye confirm karta hai ki evaluation window ke features identical hain.
(Future forecast ke features test period ke actuals use kar sakte hain, wo legit hai, kyunki wo tab tak known hote hain.)

In [ ]:
def check_no_leak(first_hidden_col, eval_cols, label):
    # first_hidden_col se aage ke actual sales NaN karo; eval_cols ke features nahi badalne chahiye
    S_masked = S_all.copy()
    S_masked[:, first_hidden_col:] = np.nan
    F_masked = build_features(S_masked, all_dates)
    bad = [k for k in FEATS
           if not np.allclose(FEATS[k][:, eval_cols], F_masked[k][:, eval_cols],
                              equal_nan=True, rtol=1e-5, atol=1e-5)]
    assert not bad, f"LEAKAGE in features: {bad}"
    print(f"OK ({label}): {len(FEATS)} features me se koi bhi {label} ke actual sales use nahi karta")

check_no_leak(T_TEST, test_idx, "test period")
check_no_leak(T_VAL, val_idx, "validation period")

In [ ]:
store_cats = sorted(SID["store"].unique())
item_cats = sorted(SID["item"].unique())

def make_rows(col_idx):
    # Long format rows (series x selected days) banata hai
    k = len(col_idx)
    data = {name: M[:, col_idx].ravel() for name, M in FEATS.items()}
    X = pd.DataFrame(data)
    X["store"] = pd.Categorical(np.repeat(SID["store"].values, k), categories=store_cats)
    X["item"] = pd.Categorical(np.repeat(SID["item"].values, k), categories=item_cats)
    y = S_all[:, col_idx].ravel()
    return X[FEATURE_NAMES], y

X_tr, y_tr = make_rows(train_idx)
X_va, y_va = make_rows(val_idx)
X_te, y_te = make_rows(test_idx)
print("Rows -> train:", X_tr.shape, "| val:", X_va.shape, "| test:", X_te.shape)

## 8. Baselines (store-item daily level)

Fancy model ki value tab hi pata chalti hai jab simple baselines se compare karo.

1. **Seasonal Naive:** pichle saal ka same weekday ka sales (364 din pehle)
2. **Smoothed Seasonal Naive:** pichle saal ke same weekday ke 5 values ka average (+/- 2 hafte)
3. **Smoothed SNaive x Growth:** upar wala, jisko recent 90 din ke YoY growth se scale kiya

In [ ]:
test_cols = slice(T_TEST, N_HIST)

# 1) Seasonal naive
F_snaive = FEATS["lag_364"][:, test_cols].astype("float64")
record("Seasonal Naive (t-364)", F_snaive)

# 2) Smoothed seasonal naive
F_snaive_s = FEATS["ya_5"][:, test_cols].astype("float64")
record("Smoothed Seasonal Naive", F_snaive_s)

# 3) Smoothed seasonal naive x growth (growth sirf test se pehle ke 90 din se: leak-free)
recent = S_all[:, T_TEST - HORIZON:T_TEST].sum(axis=1)
prev = S_all[:, T_TEST - HORIZON - 364:T_TEST - 364].sum(axis=1)
growth = np.clip(np.where(prev > 0, recent / prev, 1.0), 0.7, 1.5)
F_snaive_g = F_snaive_s * growth[:, None]
record("Smoothed SNaive x Growth", F_snaive_g)
print("\nMedian growth factor:", round(float(np.median(growth)), 3))

## 9. Classical Model: Holt-Winters (ETS) on Total Sales

Total daily sales pe weekly seasonality wala ETS. Ye sirf **total level** pe hai (store-item level pe 500 alag models bahut slow aur noisy hote).
Iska fayda: dikhata hai ki classical models yearly seasonality aur growth ko kitna handle karte hain.

In [ ]:
total_hist = S_hist.sum(axis=0)
try:
    ets = ExponentialSmoothing(total_hist[:T_TEST], trend="add", damped_trend=True,
                               seasonal="mul", seasonal_periods=7,
                               initialization_method="estimated").fit()
    ets_name = "Holt-Winters ETS (mul, weekly)"
except Exception as e:
    print("Multiplicative fit fail hua, additive use kar rahe hain:", e)
    ets = ExponentialSmoothing(total_hist[:T_TEST], trend="add", damped_trend=True,
                               seasonal="add", seasonal_periods=7,
                               initialization_method="estimated").fit()
    ets_name = "Holt-Winters ETS (add, weekly)"

ets_total = np.asarray(ets.forecast(HORIZON))
actual_total = A_test.sum(axis=0)
ets_row = {"Model": ets_name, "SMAPE": np.nan, "WAPE %": np.nan, "MAE": np.nan, "RMSE": np.nan,
           "Total-level WAPE %": wape(actual_total, ets_total)}
results.append(ets_row)
print(f"{ets_name}: total-level WAPE {ets_row['Total-level WAPE %']:.2f}% (store-item level metrics N/A)")

## 10. Global LightGBM

Ek hi model saari 500 series ke liye. Target `log1p(sales)` hai (skew kam karta hai, SMAPE ke closer).
Rounds ki sankhya **validation** se (early stopping) choose hoti hai.

In [ ]:
def to_dataset(X, y, reference=None):
    return lgb.Dataset(X, label=np.log1p(y), categorical_feature=["store", "item"],
                       reference=reference, free_raw_data=False)

dtrain = to_dataset(X_tr, y_tr)
dval = to_dataset(X_va, y_va, reference=dtrain)

t0 = time.time()
bst_val = lgb.train(
    LGB_PARAMS, dtrain, num_boost_round=MAX_ROUNDS,
    valid_sets=[dtrain, dval], valid_names=["train", "valid"],
    callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(100)],
)
BEST_ITER = int(bst_val.best_iteration)
print(f"\nBest iteration: {BEST_ITER} | time: {time.time() - t0:.0f}s")

val_pred = np.expm1(bst_val.predict(X_va, num_iteration=BEST_ITER)).clip(0)
print(f"Validation SMAPE: {smape(y_va, val_pred):.2f} | WAPE: {wape(y_va, val_pred):.2f}%")

In [ ]:
# Final fit: train + validation pe, same number of rounds
X_trva = pd.concat([X_tr, X_va], ignore_index=True)
y_trva = np.concatenate([y_tr, y_va])

t0 = time.time()
bst_test = lgb.train(LGB_PARAMS, to_dataset(X_trva, y_trva), num_boost_round=BEST_ITER)
print(f"Refit on train+val done in {time.time() - t0:.0f}s")

F_lgb = np.expm1(bst_test.predict(X_te)).clip(0).reshape(N_SERIES, HORIZON)
record("LightGBM (global)", F_lgb)

## 11. Model Comparison (Test Set)

Test = aakhri 90 din, jo kisi bhi training ya tuning me use nahi hue.

In [ ]:
res = pd.DataFrame(results)
display(res.round(3))

best_row = res.dropna(subset=["SMAPE"]).sort_values("SMAPE").iloc[0]
print("Best (store-item level SMAPE):", best_row["Model"])

# Improvement vs best baseline
base_best = res[res["Model"].str.contains("Naive|SNaive")]["SMAPE"].min()
lgb_smape = res.loc[res["Model"] == "LightGBM (global)", "SMAPE"].iloc[0]
print(f"LightGBM vs best baseline: {base_best:.2f} -> {lgb_smape:.2f} SMAPE "
      f"({100 * (base_best - lgb_smape) / base_best:.1f}% improvement)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
r1 = res.dropna(subset=["SMAPE"]).sort_values("SMAPE", ascending=False)
ax[0].barh(r1["Model"], r1["SMAPE"], color="#4C72B0")
for y, v in enumerate(r1["SMAPE"]): ax[0].text(v + 0.1, y, f"{v:.2f}", va="center", fontsize=9)
ax[0].set_title("Store-item daily SMAPE (lower is better)")
r2 = res.sort_values("Total-level WAPE %", ascending=False)
ax[1].barh(r2["Model"], r2["Total-level WAPE %"], color="#55A868")
for y, v in enumerate(r2["Total-level WAPE %"]): ax[1].text(v + 0.05, y, f"{v:.2f}%", va="center", fontsize=9)
ax[1].set_title("Total sales WAPE % (lower is better)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 12. Forecast vs Actual & Diagnostics

In [ ]:
# Total level: actual vs forecasts
ctx = 120
ctx_idx = np.arange(T_TEST - ctx, T_TEST)
fig, ax = plt.subplots(figsize=(14, 4.8))
ax.plot(all_dates[ctx_idx], S_all[:, ctx_idx].sum(axis=0), color="gray", lw=1.2, label="history")
ax.plot(all_dates[test_idx], actual_total, color="black", lw=2, label="actual (test)")
ax.plot(all_dates[test_idx], F_lgb.sum(axis=0), color="#C44E52", lw=1.8, label="LightGBM")
ax.plot(all_dates[test_idx], F_snaive_g.sum(axis=0), color="#4C72B0", lw=1.2, ls="--", label="Smoothed SNaive x Growth")
ax.plot(all_dates[test_idx], ets_total, color="#55A868", lw=1.2, ls=":", label="Holt-Winters ETS")
ax.axvline(all_dates[T_TEST], color="k", ls="--", alpha=0.4)
ax.set_title("Total daily sales: 90-day forecast vs actual")
ax.legend(ncol=3, loc="upper left"); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "forecast_total.png"), dpi=150); plt.show()

In [ ]:
# Per-series SMAPE (LightGBM) -> best / median / worst series
per_series = np.array([smape(A_test[i], F_lgb[i]) for i in range(N_SERIES)])
order = np.argsort(per_series)
picks = {"best": order[0], "median": order[len(order) // 2], "worst": order[-1], "2nd worst": order[-2]}

fig, axes = plt.subplots(2, 2, figsize=(15, 7))
for ax, (lab, i) in zip(axes.ravel(), picks.items()):
    hist_idx = np.arange(T_TEST - 60, T_TEST)
    ax.plot(all_dates[hist_idx], S_all[i, hist_idx], color="gray", lw=1, label="history")
    ax.plot(all_dates[test_idx], A_test[i], color="black", lw=1.4, label="actual")
    ax.plot(all_dates[test_idx], F_lgb[i], color="#C44E52", lw=1.4, label="LightGBM")
    ax.set_title(f"{lab}: store {SID.loc[i, 'store']}, item {SID.loc[i, 'item']} (SMAPE {per_series[i]:.1f})")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[0, 0].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "forecast_series.png"), dpi=150); plt.show()

print(f"Series SMAPE spread -> min {per_series.min():.1f} | median {np.median(per_series):.1f} | max {per_series.max():.1f}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 8))

# Error vs horizon day
h_smape = [smape(A_test[:, h], F_lgb[:, h]) for h in range(HORIZON)]
ax[0, 0].plot(range(1, HORIZON + 1), h_smape, color="#C44E52")
ax[0, 0].set_title("SMAPE by forecast horizon (day 1..90)"); ax[0, 0].set_xlabel("days ahead")

# Error by weekday
dows = all_dates[test_idx].dayofweek
d_smape = [smape(A_test[:, dows == d], F_lgb[:, dows == d]) for d in range(7)]
ax[0, 1].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], d_smape, color="#4C72B0")
ax[0, 1].set_title("SMAPE by weekday")

# Error by store
stores_u = sorted(SID["store"].unique())
s_smape = [smape(A_test[(SID["store"] == s).values], F_lgb[(SID["store"] == s).values]) for s in stores_u]
ax[1, 0].bar([str(s) for s in stores_u], s_smape, color="#55A868")
ax[1, 0].set_title("SMAPE by store"); ax[1, 0].set_xlabel("store")

# Bias by store (forecast vs actual total)
bias = [100 * (F_lgb[(SID["store"] == s).values].sum() / A_test[(SID["store"] == s).values].sum() - 1) for s in stores_u]
ax[1, 1].bar([str(s) for s in stores_u], bias, color=["#C44E52" if b > 0 else "#4C72B0" for b in bias])
ax[1, 1].axhline(0, color="k", lw=0.8)
ax[1, 1].set_title("Bias by store (% over/under-forecast)"); ax[1, 1].set_xlabel("store")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "diagnostics.png"), dpi=150); plt.show()

overall_bias = 100 * (F_lgb.sum() / A_test.sum() - 1)
print(f"Overall bias: {overall_bias:+.2f}% (positive = over-forecast)")

## 13. Feature Importance

In [ ]:
imp = pd.DataFrame({"feature": bst_test.feature_name(),
                    "gain": bst_test.feature_importance(importance_type="gain")})
imp["gain %"] = 100 * imp["gain"] / imp["gain"].sum()
imp = imp.sort_values("gain", ascending=False).reset_index(drop=True)
display(imp.head(12).round(2))

plt.figure(figsize=(8, 6))
top = imp.head(20).iloc[::-1]
plt.barh(top["feature"], top["gain %"], color="#8172B2")
plt.title("LightGBM feature importance (gain %)"); plt.xlabel("% of total gain")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "feature_importance.png"), dpi=150); plt.show()

## 14. Forecast the Next 90 Days (Jan-Mar 2018)

Final model **poori history** (2013-2017) pe fit hota hai aur agle 90 din ka forecast deta hai.
Future ke features bhi leak-free hain (sab >= 91 din purane known data se).

In [ ]:
all_idx = np.arange(T_TRAIN0, N_HIST)
X_all, y_all = make_rows(all_idx)

t0 = time.time()
bst_final = lgb.train(LGB_PARAMS, to_dataset(X_all, y_all), num_boost_round=BEST_ITER)
print(f"Final model fit in {time.time() - t0:.0f}s")

X_fut, _ = make_rows(future_idx)
F_future = np.expm1(bst_final.predict(X_fut)).clip(0).reshape(N_SERIES, HORIZON)

forecast = pd.DataFrame({
    "date": np.tile(all_dates[future_idx].values, N_SERIES),
    "store": np.repeat(SID["store"].values, HORIZON),
    "item": np.repeat(SID["item"].values, HORIZON),
    "forecast": F_future.ravel().round(2),
})
forecast.to_csv(os.path.join(OUT_DIR, "forecast_next_90_days.csv"), index=False)
print("Forecast rows:", len(forecast), "|", forecast["date"].min().date(), "->", forecast["date"].max().date())
display(forecast.head())

# Plot: last 180 din history + forecast (total)
hist_show = np.arange(N_HIST - 180, N_HIST)
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(all_dates[hist_show], S_all[:, hist_show].sum(axis=0), color="gray", lw=1.2, label="history")
ax.plot(all_dates[future_idx], F_future.sum(axis=0), color="#C44E52", lw=1.8, label="forecast (next 90 days)")
ax.axvline(all_dates[N_HIST], color="k", ls="--", alpha=0.4)
ax.set_title("Total daily sales: history and 90-day forecast")
ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "forecast_future.png"), dpi=150); plt.show()

# Competition test.csv mil jaye toh submission file bhi banate hain
sub_path = None
for root in SEARCH_ROOTS:
    for dirpath, _, files in os.walk(root):
        if "test.csv" in files:
            cols = [c.lower() for c in pd.read_csv(os.path.join(dirpath, "test.csv"), nrows=0).columns]
            if {"id", "date", "store", "item"} <= set(cols):
                sub_path = os.path.join(dirpath, "test.csv")
                break
    if sub_path: break

if sub_path:
    test_df = pd.read_csv(sub_path)
    test_df.columns = [c.lower() for c in test_df.columns]
    test_df["date"] = pd.to_datetime(test_df["date"])
    sub = test_df.merge(forecast, on=["date", "store", "item"], how="left")
    sub = sub[["id", "forecast"]].rename(columns={"forecast": "sales"})
    print("Submission NaN rows:", int(sub["sales"].isna().sum()))
    sub.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
    print("submission.csv saved:", sub.shape)
else:
    print("test.csv nahi mila, submission skip.")

## 15. Save Artifacts

In [ ]:
bst_final.save_model(os.path.join(OUT_DIR, "lgbm_sales_model.txt"))

meta = {
    "horizon": HORIZON, "min_lag": MIN_LAG, "lags": LAGS, "roll_windows": ROLL_WINDOWS,
    "features": FEATURE_NAMES, "best_iteration": BEST_ITER, "lgb_params": LGB_PARAMS,
    "train_range": [str(all_dates[T_TRAIN0].date()), str(hist_dates[-1].date())],
    "forecast_range": [str(all_dates[future_idx[0]].date()), str(all_dates[future_idx[-1]].date())],
}
with open(os.path.join(OUT_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

res.round(4).to_csv(os.path.join(OUT_DIR, "results.csv"), index=False)
imp.round(4).to_csv(os.path.join(OUT_DIR, "feature_importance.csv"), index=False)

# Reload check
reloaded = lgb.Booster(model_file=os.path.join(OUT_DIR, "lgbm_sales_model.txt"))
diff = np.abs(reloaded.predict(X_fut) - bst_final.predict(X_fut)).max()
print(f"Reload check: max prediction difference = {diff:.2e}")
assert diff < 1e-6

print("\nSaved files:")
for fn in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, fn)
    if os.path.isfile(p) and fn.endswith((".png", ".csv", ".json", ".txt")):
        print(f" - {fn:30s} {os.path.getsize(p) / 1024:.0f} KB")

In [ ]:
# README ke liye results table (markdown)
print("| Model | SMAPE | WAPE % | MAE | RMSE | Total-level WAPE % |")
print("|---|---|---|---|---|---|")
for _, r in res.iterrows():
    f = lambda v, d=2: "-" if pd.isna(v) else f"{v:.{d}f}"
    print(f"| {r['Model']} | {f(r['SMAPE'])} | {f(r['WAPE %'])} | {f(r['MAE'])} | {f(r['RMSE'])} | {f(r['Total-level WAPE %'])} |")

print(f"\nSeries: {N_SERIES} ({df['store'].nunique()} stores x {df['item'].nunique()} items) | "
      f"history: {hist_dates[0].date()} -> {hist_dates[-1].date()} | horizon: {HORIZON} days")
print(f"LightGBM best iteration: {BEST_ITER} | validation SMAPE: {smape(y_va, val_pred):.2f}")
print(f"Overall bias on test: {overall_bias:+.2f}%")
print("Top features:", ", ".join(imp['feature'].head(5)))

## ✅ Summary

- Time-based split (validation 90d + test 90d), random split nahi
- 3 baselines + Holt-Winters ETS + global LightGBM
- Features sab **>= 91 din purane** hain aur **leakage assert** se verify hua
- SMAPE / WAPE / MAE / RMSE, plus horizon, weekday, store aur bias diagnostics
- Final model se next 90 din ka forecast (`forecast_next_90_days.csv`)

**Next:** Streamlit forecasting dashboard + GitHub README + project zip (Step 3)